# Heart Disease Prediction
Carter Deslauriers

Predict `heart_disease` (0/1) from screening measurements using tree-based models.

## 1. Load the data

In [1]:
import numpy as np
import pandas as pd

train = pd.read_csv("../train.csv")
test = pd.read_csv("../test.csv")
print(train.shape, test.shape)
train.head()

(34300, 18) (14700, 17)


,patient_id,age_years,sex,height_in,weight_lb,systolic_bp,diastolic_bp,cholesterol_level,glucose_level,smoker,alcohol_use,physically_active,waist_cm,resting_hr,family_history,sleep_hours,clinic_site,heart_disease
0,200003,69,F,54.7,159.1,145,106,above_normal,normal,0,0,1,129.0,74.0,1.0,6.5,site_C,1
1,200025,64,F,68.1,168.9,153,90,well_above_normal,well_above_normal,0,0,1,87.0,68.0,0.0,6.5,site_E,1
2,200053,55,F,59.3,177.6,159,101,normal,normal,0,0,1,122.0,78.0,1.0,8.0,site_A,1
3,200116,50,F,64.4,142.2,99,79,normal,normal,0,0,1,84.0,74.0,0.0,5.0,site_B,0
4,200124,41,M,67.8,154.5,102,58,normal,normal,0,0,0,71.0,75.0,0.0,6.0,site_D,0


## 2. Explore

The target is balanced, so always guessing one class scores about 50%.

In [2]:
train["heart_disease"].value_counts(normalize=True)

heart_disease
0    0.501458
1    0.498542
Name: proportion, dtype: float64

In [3]:
train.describe().T

,count,mean,std,min,25%,50%,75%,max
patient_id,34300.0,599529.023644,230828.164248,200003.0,399300.5,600322.5,799278.25,999986.0
age_years,34300.0,53.257726,7.851035,26.0,48.0,54.0,59.00,79.0
height_in,34300.0,64.713787,3.770509,24.2,62.3,64.7,67.10,101.3
weight_lb,33256.0,163.786159,33.519118,52.2,141.1,159.8,182.40,441.1
systolic_bp,34300.0,128.908746,152.970188,-135.0,114.0,124.0,138.00,14010.0
diastolic_bp,34300.0,96.828047,199.622045,-71.0,75.0,81.0,88.00,11002.0
smoker,34300.0,0.128426,0.334568,0.0,0.0,0.0,0.00,1.0
alcohol_use,34300.0,0.098426,0.297894,0.0,0.0,0.0,0.00,1.0
physically_active,34300.0,0.775539,0.417233,0.0,1.0,1.0,1.00,1.0
waist_cm,32665.0,93.799357,20.471411,55.0,80.0,91.0,105.00,170.0


## 3. Missing values

In [4]:
# Count missing values per column, only showing columns that have some
missing = train.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
pd.DataFrame({"n_missing": missing, "pct": (missing / len(train) * 100).round(1)})

,n_missing,pct
family_history,2028,5.9
sleep_hours,1747,5.1
waist_cm,1635,4.8
resting_hr,1313,3.8
weight_lb,1044,3.0
cholesterol_level,688,2.0


In [5]:
# How many rows would we lose if we dropped every row with a missing value?
n_drop = train.isna().any(axis=1).sum()
print(f"{n_drop} rows ({n_drop / len(train):.0%}) have at least one missing value")

7619 rows (22%) have at least one missing value


## 4. Recording errors

Realistic ranges: systolic 80-220, diastolic 40-140, height 48-84 in, weight >= 80 lb.

In [6]:
print("Systolic out of range: ", ((train.systolic_bp < 80) | (train.systolic_bp > 220)).sum())
print("Diastolic out of range:", ((train.diastolic_bp < 40) | (train.diastolic_bp > 140)).sum())
print("Diastolic > systolic:  ", (train.diastolic_bp > train.systolic_bp).sum())
print("Height out of range:   ", ((train.height_in < 48) | (train.height_in > 84)).sum())
print("Weight under 80 lb:    ", (train.weight_lb < 80).sum())

Systolic out of range:  203
Diastolic out of range: 511
Diastolic > systolic:   603
Height out of range:    45
Weight under 80 lb:     35


### Cleaning rules

1. Negative values: drop the minus sign (typo).
2. Values too large: drop trailing digits until in range (e.g. 14010 -> 140).
3. Diastolic > systolic: swap them (entered in the wrong columns).
4. Anything still out of range: set to missing, handled with the other missing values.

These rules only look at one row at a time, so it is safe to apply them to train and test before splitting.

In [7]:
def fix_digits(x, high):
    # Drop trailing digits until the value is no bigger than `high`
    if pd.isna(x):
        return x
    x = abs(x)
    while x > high:
        x = x // 10
    return x

def clean(df):
    df = df.copy()
    df["systolic_bp"] = df["systolic_bp"].apply(fix_digits, high=220)
    df["diastolic_bp"] = df["diastolic_bp"].apply(fix_digits, high=140)

    # Swap when the bottom number is bigger than the top
    swap = df["diastolic_bp"] > df["systolic_bp"]
    df.loc[swap, ["systolic_bp", "diastolic_bp"]] = df.loc[swap, ["diastolic_bp", "systolic_bp"]].values

    # Anything we can't reason about becomes missing
    df.loc[(df.systolic_bp < 80) | (df.systolic_bp > 220), "systolic_bp"] = np.nan
    df.loc[(df.diastolic_bp < 40) | (df.diastolic_bp > 140), "diastolic_bp"] = np.nan
    df.loc[(df.height_in < 48) | (df.height_in > 84), "height_in"] = np.nan
    df.loc[df.weight_lb < 80, "weight_lb"] = np.nan
    return df

train_c = clean(train)
test_c = clean(test)
train_c[["systolic_bp", "diastolic_bp", "height_in", "weight_lb"]].describe().T

,count,mean,std,min,25%,50%,75%,max
systolic_bp,34192.0,127.037377,18.573884,80.0,115.0,125.0,138.00,218.0
diastolic_bp,34164.0,81.498507,11.044705,41.0,75.0,81.0,88.00,140.0
height_in,34255.0,64.745774,3.642374,48.2,62.3,64.7,67.15,82.6
weight_lb,33221.0,163.881554,33.406863,80.1,141.2,159.9,182.50,441.1


In [8]:
# How many values did cleaning turn into missing?
for col in ["systolic_bp", "diastolic_bp", "height_in", "weight_lb"]:
    print(col, train_c[col].isna().sum() - train[col].isna().sum())

systolic_bp 108
diastolic_bp 136
height_in 45
weight_lb 35


## 5. Encode categories

Cholesterol and glucose are ordered (normal < above < well above), so they become 0/1/2. Missing stays missing for now.

In [9]:
levels = {"normal": 0, "above_normal": 1, "well_above_normal": 2}

def encode(df):
    df = df.copy()
    df["cholesterol_level"] = df["cholesterol_level"].map(levels)
    df["glucose_level"] = df["glucose_level"].map(levels)
    df["sex"] = (df["sex"] == "M").astype(int)
    df = pd.get_dummies(df, columns=["clinic_site"], dtype=int)
    return df

train_e = encode(train_c)
test_e = encode(test_c)

X = train_e.drop(columns=["patient_id", "heart_disease"])
y = train_e["heart_disease"]
X_test = test_e.drop(columns=["patient_id"])
X.head()

,age_years,sex,height_in,weight_lb,systolic_bp,diastolic_bp,cholesterol_level,glucose_level,smoker,alcohol_use,...,waist_cm,resting_hr,family_history,sleep_hours,clinic_site_site_A,clinic_site_site_B,clinic_site_site_C,clinic_site_site_D,clinic_site_site_E,clinic_site_site_F
0,69,0,54.7,159.1,145.0,106.0,1.0,0,0,0,...,129.0,74.0,1.0,6.5,0,0,1,0,0,0
1,64,0,68.1,168.9,153.0,90.0,2.0,2,0,0,...,87.0,68.0,0.0,6.5,0,0,0,0,1,0
2,55,0,59.3,177.6,159.0,101.0,0.0,0,0,0,...,122.0,78.0,1.0,8.0,1,0,0,0,0,0
3,50,0,64.4,142.2,99.0,79.0,0.0,0,0,0,...,84.0,74.0,0.0,5.0,0,1,0,0,0,0
4,41,1,67.8,154.5,102.0,58.0,0.0,0,0,0,...,71.0,75.0,0.0,6.0,0,0,0,1,0,0


## 6. Validation setup

- 5-fold cross-validation on `train.csv` only, stratified so each fold keeps the 50/50 class balance.
- Metric: accuracy (what Kaggle scores).
- Imputation lives inside a pipeline, so in each fold the medians are computed from the 4 training parts only. The validation fold is never used to fit anything.

In [10]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# 5 folds, shuffled, each fold keeps the 50/50 balance
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def make_model(estimator):
    # Step 1: fill blanks with the median and add a 0/1 "was missing" column (option B)
    # Step 2: the tree model
    return Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("model", estimator),
    ])

def cv_score(estimator):
    scores = cross_val_score(make_model(estimator), X, y, cv=cv, scoring="accuracy")
    return scores.mean(), scores.std()

## 7. Baseline: a single decision tree

In [11]:
from sklearn.tree import DecisionTreeClassifier

for depth in [None, 5]:
    mean, std = cv_score(DecisionTreeClassifier(max_depth=depth, random_state=42))
    print(f"max_depth={depth}: {mean:.4f} (+/- {std:.4f})")

max_depth=None: 0.6173 (+/- 0.0043)


max_depth=5: 0.7118 (+/- 0.0032)


## 8. Hyperparameter tuning

Every model uses the same helper: grid search over the listed values, 5-fold CV, accuracy.
`tune` prints the best settings and CV score, and returns the fitted search.

In [12]:
results = {}  # model name -> best CV accuracy

def tune(name, estimator, grid):
    # Add the "model__" prefix so the grid applies to the model step of the pipeline
    grid = {"model__" + k: v for k, v in grid.items()}
    search = GridSearchCV(make_model(estimator), grid, cv=cv, scoring="accuracy", n_jobs=-1)
    search.fit(X, y)
    n_combos = len(search.cv_results_["params"])
    best = {k.replace("model__", ""): v for k, v in search.best_params_.items()}
    print(f"{name}: tried {n_combos} combinations")
    print(f"Best settings: {best}")
    print(f"Best CV accuracy: {search.best_score_:.4f}")
    results[name] = search.best_score_
    return search

### Decision tree
Baseline: one simple, explainable tree.

In [13]:
dt = tune("Decision tree", DecisionTreeClassifier(random_state=42), {
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_leaf": [1, 20, 50, 100],
})

Decision tree: tried 20 combinations
Best settings: {'max_depth': 7, 'min_samples_leaf': 100}
Best CV accuracy: 0.7148


### Random forest
Many deep trees on random rows and columns vote; averaging reduces variance.

In [14]:
from sklearn.ensemble import RandomForestClassifier

rf = tune("Random forest", RandomForestClassifier(random_state=42), {
    "n_estimators": [200, 400],
    "max_depth": [8, 12, None],
    "min_samples_leaf": [1, 10, 30],
})

Random forest: tried 18 combinations
Best settings: {'max_depth': 12, 'min_samples_leaf': 30, 'n_estimators': 400}
Best CV accuracy: 0.7255


### Extra trees
Like a random forest but with random split points, for even more variety between trees.

In [15]:
from sklearn.ensemble import ExtraTreesClassifier

et = tune("Extra trees", ExtraTreesClassifier(random_state=42), {
    "n_estimators": [200, 400],
    "max_depth": [8, 12, None],
    "min_samples_leaf": [1, 10, 30],
})

Extra trees: tried 18 combinations
Best settings: {'max_depth': None, 'min_samples_leaf': 10, 'n_estimators': 400}
Best CV accuracy: 0.7186


### Gradient boosting (LightGBM)
Trees built one after another, each fixing the previous ones' mistakes. Usually strongest on tabular data.

In [16]:
from lightgbm import LGBMClassifier

gb = tune("LightGBM", LGBMClassifier(random_state=42, verbose=-1), {
    "n_estimators": [200, 500],
    "learning_rate": [0.03, 0.1],
    "num_leaves": [15, 31],
    "min_child_samples": [20, 100],
})

LightGBM: tried 16 combinations
Best settings: {'learning_rate': 0.03, 'min_child_samples': 100, 'n_estimators': 200, 'num_leaves': 15}
Best CV accuracy: 0.7257


### XGBoost
The most widely used gradient boosting library; often the top model on tabular data in published benchmarks.

In [17]:
from xgboost import XGBClassifier

xgb = tune("XGBoost", XGBClassifier(random_state=42), {
    "n_estimators": [200, 500],
    "learning_rate": [0.03, 0.1],
    "max_depth": [3, 5],
    "subsample": [0.8, 1.0],
})

XGBoost: tried 16 combinations
Best settings: {'learning_rate': 0.03, 'max_depth': 3, 'n_estimators': 500, 'subsample': 0.8}
Best CV accuracy: 0.7265


### Comparison

In [18]:
pd.Series(results).sort_values(ascending=False).round(4)

XGBoost          0.7265
LightGBM         0.7257
Random forest    0.7255
Extra trees      0.7186
Decision tree    0.7148
dtype: float64

## 9. Final model

GridSearchCV automatically refits the best settings on **all** of train.csv once tuning is done,
so `xgb.best_estimator_` is the final model, trained on every training row.

In [19]:
final_model = xgb.best_estimator_

submission = pd.DataFrame({
    "patient_id": test["patient_id"],
    "heart_disease": final_model.predict(X_test),
})
submission.to_csv("submission.csv", index=False)
print(submission.shape)
print(submission["heart_disease"].value_counts(normalize=True))
submission.head()

(14700, 2)
heart_disease
0    0.525986
1    0.474014
Name: proportion, dtype: float64


,patient_id,heart_disease
0,200072,0
1,200191,0
2,200205,1
3,200250,0
4,200334,1
